# Inter-Annotator-Agreement: Figurenebene (NER)

**Ordnerkontext:** liegt in `00_werkzeuge/`, liest aus
`03_annotation_laufend/ner/tranche<N>/` sowie aus `02_quelldaten/`,
schreibt nach `05_ergebnisse/iaa/`.

## Konfiguration


In [ ]:
from pathlib import Path

TRANCHE = 2
BASIS_ARBEIT = (Path("..") / "03_annotation_laufend" / "ner"
                / f"tranche{TRANCHE}")
BASIS_QUELL  = Path("..") / "02_quelldaten"
CONLL        = BASIS_QUELL / "goldstandard_v1.conll"
AUSGABE      = Path("..") / "05_ergebnisse" / "iaa"

LAUF_ETIKETT = "vor Adjudikation"

RICHTLINIE_VERSION = "1.18"
BOOTSTRAP_REPS, SEED = 2000, 42

AUSGABE.mkdir(parents=True, exist_ok=True)
assert CONLL.exists(), f"{CONLL} nicht gefunden."

gefunden = []
for endung in ("xlsx", "ods"):
    gefunden += [q for q in BASIS_ARBEIT.glob(f"*_ner_tranche*.{endung}")
                 if not q.name.startswith(("~$", "."))]
mappen = {}
for q in sorted(gefunden, key=lambda x: x.name):
    person = q.name.split("_")[0]
    if person in mappen:
        raise ValueError(
            f"Fuer {person} liegen zwei Dateien vor: "
            f"{mappen[person].name} und {q.name}. Bitte eine entfernen.")
    mappen[person] = q
assert len(mappen) >= 2, f"Mindestens zwei Mappen noetig: {list(mappen)}"

LAUF_NR = len(sorted(AUSGABE.glob(f"iaa_ner_t{TRANCHE}_lauf*.json"))) + 1
print(f"Tranche {TRANCHE} | Bearbeitende ({len(mappen)}):")
for person, q in mappen.items():
    print(f"  {person:<5} {q.name}")
print(f"Dieser Durchlauf: Lauf {LAUF_NR} ({LAUF_ETIKETT})")

Tranche 2 | Bearbeitende (4):
  A1    A1_ner_tranche2_final.xlsx
  A3    A3_ner_tranche2_final.ods
  A7    A7_ner_tranche2_final.xlsx
  A8    A8_ner_tranche2_final.xlsx
Dieser Durchlauf: Lauf 1 (vor Adjudikation)


## Schritt 0b — Lesepakete prüfen


In [ ]:
import importlib.util

benoetigt = {"xlsx": ("openpyxl", "openpyxl"), "ods": ("odf", "odfpy")}
fehlt = []
for endung, (modul, paket) in benoetigt.items():
    vorhanden = any(q.suffix == f".{endung}" for q in mappen.values())
    if vorhanden and importlib.util.find_spec(modul) is None:
        fehlt.append((endung, paket))

if fehlt:
    print("FEHLENDE PAKETE\n")
    for endung, paket in fehlt:
        print(f"  Fuer die vorliegenden .{endung}-Dateien wird '{paket}' benoetigt.")
    print("\nInstallation:")
    print("  pip install " + " ".join(paket for _, paket in fehlt))
    print("\nDanach den Kernel neu starten und dieses Notebook erneut ausfuehren.")
    raise SystemExit("Bitte die genannten Pakete installieren.")
print("Lesepakete vorhanden.")


Lesepakete vorhanden.


## Schritt 1 — Maße implementieren und selbst validieren



In [ ]:
from collections import Counter
from itertools import permutations, combinations

def spannen(marken):
    ergebnis, start = set(), None
    for i, m in enumerate(marken):
        if m == "B-FIGUR":
            if start is not None:
                ergebnis.add((start, i))
            start = i
        elif m == "I-FIGUR":
            if start is None:
                start = i
        else:
            if start is not None:
                ergebnis.add((start, i))
                start = None
    if start is not None:
        ergebnis.add((start, len(marken)))
    return ergebnis


def paar_f1(a, b):
    if not a and not b:
        return 1.0, 1.0, 1.0
    treffer = len(a & b)
    prec = treffer / len(b) if b else 0.0
    rec = treffer / len(a) if a else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    return prec, rec, f1


def prozent_agreement(einheiten):
    werte = []
    for urteile in einheiten:
        u = [x for x in urteile if x is not None]
        paare = list(permutations(range(len(u)), 2))
        werte.append(sum(1 for i, j in paare if u[i] == u[j]) / len(paare))
    return sum(werte) / len(werte)


def einheit_agreement(urteile):
    u = [x for x in urteile if x is not None]
    paare = list(permutations(range(len(u)), 2))
    return sum(1 for i, j in paare if u[i] == u[j]) / len(paare)


def fleiss_kappa(einheiten):
    umfaenge = {len([x for x in u if x is not None]) for u in einheiten}
    if len(umfaenge) > 1:
        raise ValueError(
            "Fleiss' kappa verlangt gleich viele Urteile je Einheit; "
            f"vorgefunden: {sorted(umfaenge)}. Mappen pruefen.")
    p_o = prozent_agreement(einheiten)
    zaehler, gesamt = Counter(), 0
    for urteile in einheiten:
        for x in (v for v in urteile if v is not None):
            zaehler[x] += 1
            gesamt += 1
    p_e = sum((v / gesamt) ** 2 for v in zaehler.values())
    return ((p_o - p_e) / (1 - p_e) if p_e < 1 else float("nan")), p_o, p_e


def kategorienverteilung(einheiten):
    zaehler, gesamt = Counter(), 0
    for urteile in einheiten:
        for x in (v for v in urteile if v is not None):
            zaehler[x] += 1
            gesamt += 1
    return {k: v / gesamt for k, v in zaehler.most_common()}


def lesehilfe_kappa(einheiten, kappa, beobachtet, erwartet, einrueckung=""):
    vert = kategorienverteilung(einheiten)
    gezeigt = list(vert.items())[:6]
    rest = len(vert) - len(gezeigt)
    print(f"{einrueckung}Kategorien ({len(vert)}): "
          + " | ".join(f"{k} {100 * v:.1f}%" for k, v in gezeigt)
          + (f" | ... und {rest} weitere" if rest > 0 else ""))
    print(f"{einrueckung}erwartet = Summe der quadrierten Anteile = "
          f"{erwartet:.4f}")
    print(f"{einrueckung}kappa = ({beobachtet:.4f} - {erwartet:.4f}) / "
          f"(1 - {erwartet:.4f}) = {beobachtet - erwartet:.4f} / "
          f"{1 - erwartet:.4f} = {kappa:.4f}")


def fleiss_gegenprobe(einheiten):
    try:
        import numpy as np
        from statsmodels.stats.inter_rater import (aggregate_raters,
                                                   fleiss_kappa as sm_fleiss)
    except ImportError:
        return None
    tabelle, _ = aggregate_raters(np.array(einheiten))
    return float(sm_fleiss(tabelle, method="fleiss"))


# Anker 1: Fleiss, 30 Einheiten und 6 Bearbeitende, wie im Testfall der Bibliothek DKPro Agreement
# Sollwerte: beobachtet 0,5556 | erwartet 0,2201 | kappa 0,430
referenz = [(4, 4, 4, 4, 4, 4), (2, 2, 2, 5, 5, 5), (2, 3, 3, 3, 3, 5),
            (5, 5, 5, 5, 5, 5), (2, 2, 2, 4, 4, 4), (1, 1, 3, 3, 3, 3),
            (3, 3, 3, 3, 5, 5), (1, 1, 3, 3, 3, 4), (1, 1, 4, 4, 4, 4),
            (5, 5, 5, 5, 5, 5), (1, 4, 4, 4, 4, 4), (1, 2, 4, 4, 4, 4),
            (2, 2, 2, 3, 3, 3), (1, 4, 4, 4, 4, 4), (2, 2, 4, 4, 4, 5),
            (3, 3, 3, 3, 3, 5), (1, 1, 1, 4, 5, 5), (1, 1, 1, 1, 1, 2),
            (2, 2, 4, 4, 4, 4), (1, 3, 3, 5, 5, 5), (5, 5, 5, 5, 5, 5),
            (2, 4, 4, 4, 4, 4), (2, 2, 4, 5, 5, 5), (1, 1, 4, 4, 4, 4),
            (1, 4, 4, 4, 4, 5), (2, 2, 2, 2, 2, 4), (1, 1, 1, 1, 5, 5),
            (2, 2, 4, 4, 4, 4), (1, 3, 3, 3, 3, 3), (5, 5, 5, 5, 5, 5)]
fk, po, pe = fleiss_kappa(referenz)
print("Selbstvalidierung")
print(f"  Fleiss (Soll 0,5556 / 0,2201 / 0,430): "
      f"{po:.4f} / {pe:.4f} / {fk:.4f}")
for wert, soll in ((po, 0.5556), (pe, 0.2201), (fk, 0.430)):
    assert abs(wert - soll) < 0.001

fremd = fleiss_gegenprobe(referenz)
if fremd is None:
    print("  Gegenprobe uebersprungen: statsmodels nicht installiert")
else:
    assert abs(fk - fremd) < 1e-9, (fk, fremd)
    print(f"  Gegenprobe statsmodels: {fremd:.4f} — uebereinstimmend")

# Die Voraussetzung gleicher Urteilszahlen muss erkannt werden.
try:
    fleiss_kappa([(1, 1, 1), (1, 1)])
except ValueError:
    print("  Ungleiche Urteilszahlen werden erkannt")
else:
    raise AssertionError("ungleiche Urteilszahlen blieben unbemerkt")

# Anker 2 und 3: Grenzfaelle, deren Ergebnis feststeht
#   identische Reihen bei zwei Kategorien -> kappa = 1
#   Uebereinstimmung genau auf Zufallsniveau -> kappa = 0
assert abs(fleiss_kappa([("a", "a"), ("b", "b"),
                         ("a", "a"), ("b", "b")])[0] - 1.0) < 1e-9
assert abs(fleiss_kappa([("x", "x"), ("x", "y"),
                         ("y", "x"), ("y", "y")])[0]) < 1e-9
print("  Fleiss kappa: Grenzfaelle 1,0 und 0,0 reproduziert")

# Anker 4: Uebereinstimmung innerhalb einer einzelnen Einheit. Diese
# Funktion bestimmt spaeter, welche Einheiten als einstimmig gelten und
# damit, was in die Divergenz- und Kommentarlisten gelangt.
assert einheit_agreement(("a", "a", "a")) == 1.0
assert abs(einheit_agreement(("a", "a", "b")) - 1 / 3) < 1e-9
assert einheit_agreement(("a", "b", "c")) == 0.0
print("  einheit_agreement: 1,000 / 0,333 / 0,000 korrekt")

# Anker: Kategorienverteilung, aus der die erwartete Uebereinstimmung folgt
_v = kategorienverteilung([("a", "a"), ("a", "b")])
assert abs(_v["a"] - 0.75) < 1e-9 and abs(_v["b"] - 0.25) < 1e-9
assert abs(sum(x * x for x in _v.values()) - 0.625) < 1e-9
print("  kategorienverteilung: 0,75 / 0,25 korrekt")

# Anker 6: Spannenextraktion und F1 von Hand nachgerechnet
probe_a = ["O", "B-FIGUR", "I-FIGUR", "O", "B-FIGUR"]
probe_b = ["O", "B-FIGUR", "O", "O", "B-FIGUR"]
sa, sb = spannen(probe_a), spannen(probe_b)
assert sa == {(1, 3), (4, 5)}, sa
assert sb == {(1, 2), (4, 5)}, sb
# gemeinsam: (4,5) -> 1 Treffer; je 2 Spannen -> P = R = 0,5 -> F1 = 0,5
p_, r_, f_ = paar_f1(sa, sb)
assert (abs(p_ - 0.5) < 1e-9 and abs(r_ - 0.5) < 1e-9
        and abs(f_ - 0.5) < 1e-9), (p_, r_, f_)
assert paar_f1(set(), set()) == (1.0, 1.0, 1.0)
print("  Spannenextraktion und F1: Beispiel korrekt (P=R=F1=0,500)")
print()
print("Alle Anker getroffen — die Implementierung darf verwendet werden.")

Selbstvalidierung
  Fleiss (Soll 0,5556 / 0,2201 / 0,430): 0.5556 / 0.2199 / 0.4302
  Gegenprobe statsmodels: 0.4302 — uebereinstimmend
  Ungleiche Urteilszahlen werden erkannt
  Fleiss kappa: Grenzfaelle 1,0 und 0,0 reproduziert
  einheit_agreement: 1,000 / 0,333 / 0,000 korrekt
  kategorienverteilung: 0,75 / 0,25 korrekt
  Spannenextraktion und F1: Beispiel korrekt (P=R=F1=0,500)

Alle Anker getroffen — die Implementierung darf verwendet werden.


## Schritt 2 — Mappen laden und ausrichten


In [4]:
import pandas as pd

tabellen = {}
for person, pfad in mappen.items():
    df = pd.read_excel(pfad, sheet_name="NER", dtype=str).fillna("")
    df["id"] = df["id"].astype(int)
    tabellen[person] = df.sort_values("id").reset_index(drop=True)
    print(f"  {person}: {len(df)} Zeilen")

personen = list(tabellen)
basis = tabellen[personen[0]]
for person, df in tabellen.items():
    assert list(df["id"]) == list(basis["id"]), f"{person}: andere Tokenmenge."
    assert list(df["token"]) == list(basis["token"]), \
        f"{person}: anderes Token bei gleicher id."
print(f"\nAusrichtung ueber 'id' geprueft: {len(basis)} Token.")

herkunft, lauf_id, meta = {}, 0, {}
for z in CONLL.read_text(encoding="utf-8").splitlines():
    if z.startswith("#"):
        if " = " in z:
            k, w = z[1:].split(" = ", 1)
            meta[k.strip()] = w.strip()
        continue
    if not z.strip():
        continue
    herkunft[lauf_id] = (meta.get("erzaehlung", "?"), meta.get("sammlung", "?"))
    lauf_id += 1

daten = pd.DataFrame({"id": basis["id"], "token": basis["token"],
                      "satz": basis["satz"].astype(int)})
daten["erzaehlung"] = [herkunft[i][0] for i in daten["id"]]
daten["sammlung"]   = [herkunft[i][1] for i in daten["id"]]
for person in personen:
    daten[person] = (tabellen[person]["markierung"].str.strip().str.upper()
                     .replace("", "O").values)
    daten["unsicher_" + person] = (
        tabellen[person]["unsicher"].str.strip().str.lower() != "")

erlaubt = {"O", "B-FIGUR", "I-FIGUR"}
for person in personen:
    fremd = set(daten[person]) - erlaubt
    assert not fremd, f"{person} enthaelt unzulaessige Marken: {fremd}"
    n_fig = int((daten[person] != "O").sum())
    print(f"  {person}: {n_fig} Token als Figur markiert "
          f"({100 * n_fig / len(daten):.1f} %)")

  A1: 2343 Zeilen
  A3: 2343 Zeilen
  A7: 2343 Zeilen
  A8: 2343 Zeilen

Ausrichtung ueber 'id' geprueft: 2343 Token.
  A1: 116 Token als Figur markiert (5.0 %)
  A3: 121 Token als Figur markiert (5.2 %)
  A7: 120 Token als Figur markiert (5.1 %)
  A8: 116 Token als Figur markiert (5.0 %)


## Schritt 3 — Hauptmaß: paarweises F1 bei exaktem Span-Match


In [ ]:
spannen_je_person = {}
for person in personen:
    menge = set()
    for satz_nr, gruppe in daten.groupby("satz"):
        marken = list(gruppe[person])
        ids = list(gruppe["id"])
        for a, b in spannen(marken):
            menge.add((ids[a], ids[b - 1]))     # (erste id, letzte id)
    spannen_je_person[person] = menge
    print(f"  {person}: {len(menge)} Figurenspannen")

print()
print(f"{'Paar':<12}{'Spannen A':>10}{'Spannen B':>10}{'Treffer':>9}"
      f"{'Precision':>11}{'Recall':>9}{'F1':>9}")
paar_f1_werte = {}
for a, b in combinations(personen, 2):
    menge_a, menge_b = spannen_je_person[a], spannen_je_person[b]
    treffer = len(menge_a & menge_b)
    p_, r_, f_ = paar_f1(menge_a, menge_b)
    paar_f1_werte[f"{a}-{b}"] = dict(precision=p_, recall=r_, f1=f_,
                                     spannen_a=len(menge_a),
                                     spannen_b=len(menge_b),
                                     treffer=treffer)
    print(f"{a + '-' + b:<12}{len(menge_a):>10}{len(menge_b):>10}"
          f"{treffer:>9}{p_:>11.3f}{r_:>9.3f}{f_:>9.3f}")

print()
print("Lesehilfe: Precision = Treffer / Spannen B, "
      "Recall = Treffer / Spannen A,")
print("           F1 = harmonisches Mittel aus beiden.")
beispiel = list(paar_f1_werte.items())[0]
_name, _w = beispiel
print(f"  Beispiel {_name}: {_w['treffer']} / {_w['spannen_b']} = "
      f"{_w['precision']:.3f} und {_w['treffer']} / {_w['spannen_a']} = "
      f"{_w['recall']:.3f}  ->  F1 {_w['f1']:.3f}")

f1_mittel = sum(v["f1"] for v in paar_f1_werte.values()) / len(paar_f1_werte)
print()
print(f"HAUPTMASS — paarweises F1 (Mittel ueber alle Paare): {f1_mittel:.3f}")
print("Diese Zahl markiert die Obergrenze fuer die sinnvolle Bewertung des")
print("Modells: Ein Modell-F1 oberhalb dieses Wertes waere kein Beleg fuer")
print("Ueberlegenheit, sondern ein Hinweis auf einen unzuverlaessigen Massstab.")

  A1: 112 Figurenspannen
  A3: 116 Figurenspannen
  A7: 116 Figurenspannen
  A8: 112 Figurenspannen

Paar         Spannen A Spannen B  Treffer  Precision   Recall       F1
A1-A3              112       116      105      0.905    0.938    0.921
A1-A7              112       116      104      0.897    0.929    0.912
A1-A8              112       112      104      0.929    0.929    0.929
A3-A7              116       116      105      0.905    0.905    0.905
A3-A8              116       112      109      0.973    0.940    0.956
A7-A8              116       112      105      0.938    0.905    0.921

Lesehilfe: Precision = Treffer / Spannen B, Recall = Treffer / Spannen A,
           F1 = harmonisches Mittel aus beiden.
  Beispiel A1-A3: 105 / 116 = 0.905 und 105 / 112 = 0.938  ->  F1 0.921

HAUPTMASS — paarweises F1 (Mittel ueber alle Paare): 0.924
Diese Zahl markiert die Obergrenze fuer die sinnvolle Bewertung des
Modells: Ein Modell-F1 oberhalb dieses Wertes waere kein Beleg fuer
Ueberlegenh

## Schritt 4 — Fehlerarten trennen



In [6]:
def ueberlappt(s1, s2):
    return not (s1[1] < s2[0] or s2[1] < s1[0])

fehlerarten = {}
for a, b in combinations(personen, 2):
    sa, sb = spannen_je_person[a], spannen_je_person[b]
    nur_a, nur_b = sa - sb, sb - sa
    grenze_a = {s for s in nur_a if any(ueberlappt(s, t) for t in nur_b)}
    grenze_b = {s for s in nur_b if any(ueberlappt(s, t) for t in nur_a)}
    existenz = (nur_a - grenze_a) | (nur_b - grenze_b)
    fehlerarten[f"{a}-{b}"] = dict(
        uebereinstimmend=len(sa & sb),
        grenzabweichung=len(grenze_a) + len(grenze_b),
        existenzabweichung=len(existenz))

print(f"{'Paar':<12}{'gleich':>9}{'Grenze':>9}{'Existenz':>11}")
for paar, w in fehlerarten.items():
    print(f"{paar:<12}{w['uebereinstimmend']:>9}"
          f"{w['grenzabweichung']:>9}{w['existenzabweichung']:>11}")

g = sum(w["grenzabweichung"] for w in fehlerarten.values())
e = sum(w["existenzabweichung"] for w in fehlerarten.values())
print()
if g + e == 0:
    print("Keine Abweichungen.")
else:
    print(f"Von allen Abweichungen entfallen {100 * g / (g + e):.0f} % auf "
          f"Grenzfragen und {100 * e / (g + e):.0f} % auf die Frage, ob "
          f"ueberhaupt")
    print("eine Figur vorliegt.")
    if e > g:
        print("Schwerpunkt Existenz: Die Figurendefinition (R1 bis R12) "
              "bedarf der Praezisierung.")
    else:
        print("Schwerpunkt Grenzen: Die Regeln zur Spannenausdehnung "
              "(R13 folgende) bedarfen der Praezisierung.")

Paar           gleich   Grenze   Existenz
A1-A3             105        5         13
A1-A7             104        0         20
A1-A8             104        0         16
A3-A7             105        5         17
A3-A8             109        3          7
A7-A8             105        0         18

Von allen Abweichungen entfallen 12 % auf Grenzfragen und 88 % auf die Frage, ob ueberhaupt
eine Figur vorliegt.
Schwerpunkt Existenz: Die Figurendefinition (R1 bis R12) bedarf der Praezisierung.


## Schritt 5 — Tokenebene: Übereinstimmung und Fleiss' κ



In [7]:
einheiten = [tuple(r) for r in daten[personen].to_numpy()]
fk_tok, pa_tok, pe_tok = fleiss_kappa(einheiten)
daten["einheit_pa"] = [einheit_agreement(u) for u in einheiten]
daten["einstimmig"] = daten["einheit_pa"] == 1.0

print(f"Tokenebene: beobachtet {pa_tok:.3f} | erwartet {pe_tok:.3f} | "
      f"Fleiss kappa {fk_tok:.3f}")
print(f"  nicht einstimmig: {int((~daten['einstimmig']).sum())} "
      f"von {len(daten)} Token")

lesehilfe_kappa(einheiten, fk_tok, pa_tok, pe_tok, einrueckung="  ")

#Gegenprobe auf genau diesen Daten, nicht nur auf dem Referenzbeispiel
fremd_tok = fleiss_gegenprobe(einheiten)
if fremd_tok is None:
    print("  Gegenprobe uebersprungen: statsmodels nicht installiert")
else:
    assert abs(fk_tok - fremd_tok) < 1e-9, (fk_tok, fremd_tok)
    print(f"  Gegenprobe statsmodels: {fremd_tok:.3f} — uebereinstimmend")


print()
print("Nach Marke (Zuordnung nach der ersten Bearbeitung):")
for marke, teil in daten.groupby(personen[0]):
    print(f"  {marke:<9} {teil['einheit_pa'].mean():.3f}  (n={len(teil)}, "
          f"nicht einstimmig={int((~teil['einstimmig']).sum())})")

index = {}
if len(personen) > 2:
    print()
    print("Abweichungsindex (mittleres paarweises F1 mit den anderen):")
    for p in personen:
        werte = [v["f1"] for k, v in paar_f1_werte.items()
                 if p in k.split("-")]
        index[p] = sum(werte) / len(werte)
    for p, v in sorted(index.items(), key=lambda x: x[1]):
        print(f"  {p:<5} {v:.3f}")
    spanne = max(index.values()) - min(index.values())
    if spanne > 0.05:
        print(f"  Hinweis: {min(index, key=index.get)} weicht deutlich ab "
              f"(Spanne {spanne:.3f}).")
    else:
        print(f"  Kein Ausreisser erkennbar (Spanne {spanne:.3f}).")

Tokenebene: beobachtet 0.992 | erwartet 0.904 | Fleiss kappa 0.920
  nicht einstimmig: 34 von 2343 Token
  Kategorien (3): O 95.0% | B-FIGUR 4.8% | I-FIGUR 0.2%
  erwartet = Summe der quadrierten Anteile = 0.9040
  kappa = (0.9923 - 0.9040) / (1 - 0.9040) = 0.0884 / 0.0960 = 0.9200
  Gegenprobe statsmodels: 0.920 — uebereinstimmend

Nach Marke (Zuordnung nach der ersten Bearbeitung):
  B-FIGUR   0.932  (n=110, nicht einstimmig=13)
  I-FIGUR   0.750  (n=6, nicht einstimmig=3)
  O         0.996  (n=2227, nicht einstimmig=18)

Abweichungsindex (mittleres paarweises F1 mit den anderen):
  A7    0.913
  A1    0.921
  A3    0.927
  A8    0.935
  Kein Ausreisser erkennbar (Spanne 0.022).


## Schritt 6 — Unsicherheit der Schätzung


In [8]:
import numpy as np

erzaehlungen = sorted(daten["erzaehlung"].unique())
ids_je_erz = {e: set(daten[daten["erzaehlung"] == e]["id"])
              for e in erzaehlungen}

def f1_auf_teilmenge(auswahl_ids):
    werte = []
    for a, b in combinations(personen, 2):
        sa = {s for s in spannen_je_person[a] if s[0] in auswahl_ids}
        sb = {s for s in spannen_je_person[b] if s[0] in auswahl_ids}
        werte.append(paar_f1(sa, sb)[2])
    return sum(werte) / len(werte)

rng = np.random.default_rng(SEED)
proben = []
for _ in range(BOOTSTRAP_REPS):
    ziehung = rng.integers(0, len(erzaehlungen), size=len(erzaehlungen))
    auswahl = set()
    for i in ziehung:
        auswahl |= ids_je_erz[erzaehlungen[i]]
    proben.append(f1_auf_teilmenge(auswahl))
lo, hi = np.percentile(proben, [2.5, 97.5])

print(f"Hauptmass (paarweises F1): {f1_mittel:.3f}")
print(f"  Cluster-Bootstrap ueber {len(erzaehlungen)} Erzaehlungen: "
      f"[{lo:.3f}, {hi:.3f}]")
print()
print("Nach Sammlung (paarweises F1):")
nach_sammlung = {}
for s in sorted(daten["sammlung"].unique()):
    ids_s = set(daten[daten["sammlung"] == s]["id"])
    nach_sammlung[s] = f1_auf_teilmenge(ids_s)
    n_s = int((daten["sammlung"] == s).sum())
    print(f"  {s:<12} {nach_sammlung[s]:.3f}  (n={n_s} Token)")

Hauptmass (paarweises F1): 0.924
  Cluster-Bootstrap ueber 20 Erzaehlungen: [0.887, 0.964]

Nach Sammlung (paarweises F1):
  Andersen     0.904  (n=560 Token)
  Arabisch     0.946  (n=659 Token)
  Grimm        0.950  (n=597 Token)
  Japanisch    0.885  (n=527 Token)


## Schritt 7 — Divergenzen nach Strittigkeit auflisten



In [ ]:
alle_spannen = set()
for menge in spannen_je_person.values():
    alle_spannen |= menge

id2token = dict(zip(daten["id"], daten["token"]))
id2satz = dict(zip(daten["id"], daten["satz"]))
satz_text = {s: " ".join(g["token"]) for s, g in daten.groupby("satz")}
id2erz = dict(zip(daten["id"], daten["erzaehlung"]))
id2samm = dict(zip(daten["id"], daten["sammlung"]))

n_pers = len(personen)
zeilen_export = []
for spanne in sorted(alle_spannen):
    traeger = [p for p in personen if spanne in spannen_je_person[p]]
    if len(traeger) == n_pers:
        continue                       # alle einig
    start, ende = spanne
    if len(traeger) * 2 > n_pers:
        lage = "Mehrheit dafuer"
    elif len(traeger) * 2 == n_pers:
        lage = "Patt"
    else:
        lage = "Einzelmeinung"
    text = " ".join(id2token[i] for i in range(start, ende + 1))
    zeilen_export.append({
        "lage": lage, "dafuer": len(traeger), "von": n_pers,
        "id_start": start, "id_ende": ende, "text": text,
        "erzaehlung": id2erz[start], "sammlung": id2samm[start],
        "markiert_von": " ".join(traeger),
        "nicht_markiert_von": " ".join(p for p in personen
                                       if p not in traeger),
        "mehrheit": "ja" if lage == "Mehrheit dafuer" else "nein",
        "satz": satz_text[id2satz[start]],
        "entscheidung": "", "begruendung": ""})

#Nach Strittigkeit sortieren: Patt zuerst, dann Mehrheit, dann Einzelmeinung
rang = {"Patt": 0, "Mehrheit dafuer": 1, "Einzelmeinung": 2}
zeilen_export.sort(key=lambda z: (rang[z["lage"]], z["id_start"]))

from collections import Counter
verteilung = Counter(z["lage"] for z in zeilen_export)
print(f"Strittige Spannen: {len(zeilen_export)} von "
      f"{len(alle_spannen)} insgesamt genannten\n")
for lage in ("Patt", "Mehrheit dafuer", "Einzelmeinung"):
    k = verteilung.get(lage, 0)
    erklaerung = {"Patt": f"{n_pers//2} dafuer, {n_pers//2} dagegen — "
                          f"ohne Entscheidung nicht aufloesbar",
                  "Mehrheit dafuer": "Mehrheit haelt es fuer eine Figur",
                  "Einzelmeinung": "nur eine Person hat markiert"}[lage]
    print(f"  {lage:<16} {k:>3}   ({erklaerung})")

for lage in ("Patt", "Mehrheit dafuer", "Einzelmeinung"):
    teil = [z for z in zeilen_export if z["lage"] == lage]
    if not teil:
        continue
    print(f"\n{'=' * 62}\n{lage} ({len(teil)})\n{'=' * 62}")
    for z in teil[:12]:
        print(f"  '{z['text']}'  ({z['dafuer']}/{z['von']}, "
              f"id {z['id_start']}-{z['id_ende']}, {z['sammlung']})")
        print(f"      dafuer: {z['markiert_von'] or '—'} | "
              f"dagegen: {z['nicht_markiert_von'] or '—'}")
        kontext = z["satz"]
        print(f"      {kontext[:96]}{'...' if len(kontext) > 96 else ''}")
    if len(teil) > 12:
        print(f"  … {len(teil) - 12} weitere in der CSV")

ziel = AUSGABE / f"divergenzen_ner_t{TRANCHE}_lauf{LAUF_NR:02d}.csv"
pd.DataFrame(zeilen_export).to_csv(ziel, index=False, encoding="utf-8")
print(f"\nGeschrieben: {ziel}")
print("Die CSV ist nach Strittigkeit sortiert; die Patt-Faelle stehen oben.")


Strittige Spannen: 33 von 131 insgesamt genannten

  Patt               5   (2 dafuer, 2 dagegen — ohne Entscheidung nicht aufloesbar)
  Mehrheit dafuer   13   (Mehrheit haelt es fuer eine Figur)
  Einzelmeinung     15   (nur eine Person hat markiert)

Patt (5)
  'Decke'  (2/4, id 865-865, Andersen)
      dafuer: A1 A7 | dagegen: A3 A8
      Er berührte mit demselben die Decke , welche sich höher und höher dehnte und dort , wo sie berüh...
  'Wände'  (2/4, id 891-891, Andersen)
      dafuer: A1 A7 | dagegen: A3 A8
      Er berührte mit demselben die Decke , welche sich höher und höher dehnte und dort , wo sie berüh...
  'Orgel'  (2/4, id 901-901, Andersen)
      dafuer: A1 A7 | dagegen: A3 A8
      Er berührte mit demselben die Decke , welche sich höher und höher dehnte und dort , wo sie berüh...
  'Heiligen'  (2/4, id 3416-3416, Japanisch)
      dafuer: A1 A7 | dagegen: A3 A8
      Man nannte ihn nun den großen kunstverständigen Heiligen , und die Menge kam scharenweis von all...
  'H

## Schritt 8 — Kommentare der Bearbeitenden einsammeln



In [10]:
basis_index = set(daten.index)
strittige_index = {i for i, r in daten.iterrows()
                   if not r["einstimmig"]}
id_je_index = dict(zip(daten.index, daten["id"]))
token_je_index = dict(zip(daten.index, daten["token"]))

kommentare = []
for person in personen:
    spalte = tabellen[person].get("kommentar")
    if spalte is None:
        continue
    for idx, text in spalte.items():
        text = str(text).strip()
        if not text or idx not in basis_index:
            continue
        kommentare.append({
            "person": person,
            "id": int(id_je_index[idx]),
            "token": token_je_index[idx],
            "strittig": bool(idx in strittige_index),
            "kommentar": text,
        })

print(f"Kommentare insgesamt: {len(kommentare)}")
bei_strittig = [k for k in kommentare if k["strittig"]]
bei_einig    = [k for k in kommentare if not k["strittig"]]
print(f"  bei strittigen Einheiten:   {len(bei_strittig)}")
print(f"  bei einstimmigen Einheiten: {len(bei_einig)}  <- besonders pruefen")

if bei_einig:
    print()
    print("Kommentare trotz Einstimmigkeit (moegliche Regelluecke):")
    for k in bei_einig[:15]:
        print(f"  [{k['person']}] {k['token']}: {k['kommentar'][:70]}")
if bei_strittig:
    print()
    print("Kommentare bei strittigen Einheiten (Auszug):")
    for k in bei_strittig[:10]:
        print(f"  [{k['person']}] {k['token']}: {k['kommentar'][:70]}")

if kommentare:
    ziel_k = AUSGABE / f"kommentare_ner_t{TRANCHE}_lauf{LAUF_NR:02d}.csv"
    pd.DataFrame(kommentare).to_csv(ziel_k, index=False, encoding="utf-8")
    print(f"\nGeschrieben: {ziel_k}")
else:
    print("\nKeine Kommentare hinterlegt.")


Kommentare insgesamt: 41
  bei strittigen Einheiten:   21
  bei einstimmigen Einheiten: 20  <- besonders pruefen

Kommentare trotz Einstimmigkeit (moegliche Regelluecke):
  [A1] Sohn: wird ja nur angesprochen
  [A1] Brüder: Ist wissen etwas was sie tun
  [A3] Tafel: klingt für mich wie eine magische beseelte Tafel, wenn sie einen menty
  [A3] Engel: ist zwar Vergleich aber ein Vergleich mit einer bestimmten Person, da 
  [A7] Tafel: ist ja am Jammern
  [A7] Feldblume: erhält durch Gott eine Stimme und singt
  [A7] Betrüger: sollte ja sein, wie „die Räuber“ im ersten Teil
  [A7] Sklavinnen: bin mir unsicher ob zu allgemein, ich würde sagen Figur
  [A7] Zughb: vermute damit sind die Menschen gemeint
  [A7] Schar: würde sagen gehört zum fremden Ritter
  [A7] Feinde: keine Figur, weil zu allgemein gehalten
  [A7] Hunkepuus: keine Figur, auch wenn das Pferd einen speziellen Namen hat, macht es 
  [A7] Geistern: keine Figur, allgemeiner Sammelbegriff
  [A7] Eltern: eindeutig bestimmbar
  [A7

## Schritt 9 — Ergebnis sichern und einordnen


In [11]:
import json
from datetime import datetime

ergebnis = {
    "lauf": LAUF_NR, "etikett": LAUF_ETIKETT,
    "richtlinie_version": RICHTLINIE_VERSION, "tranche": TRANCHE,
    "zeitpunkt": datetime.now().isoformat(timespec="minutes"),
    "ebene": "ner", "bearbeitende": personen,
    "n_token": int(len(daten)),
    "spannen_je_person": {p: len(s) for p, s in spannen_je_person.items()},
    "hauptmass_paarweises_f1": {"mittel": float(f1_mittel),
                                "paare": paar_f1_werte},
    "cluster_bootstrap_95": [float(lo), float(hi)],
    "fehlerarten": fehlerarten,
    "tokenebene": {"prozent": float(pa_tok),
                   "fleiss_kappa": float(fk_tok),
                   "erwartet": float(pe_tok),
                   "kategorienverteilung": {
                       str(k): float(v)
                       for k, v in kategorienverteilung(einheiten).items()}},
    "abweichungsindex": {k: float(v) for k, v in index.items()},
    "nach_sammlung": {k: float(v) for k, v in nach_sammlung.items()},
    "strittige_spannen": len(zeilen_export),
}
ziel_json = AUSGABE / f"iaa_ner_t{TRANCHE}_lauf{LAUF_NR:02d}.json"
ziel_json.write_text(json.dumps(ergebnis, ensure_ascii=False, indent=2),
                     encoding="utf-8")
print(f"Geschrieben: {ziel_json}")

print()
print(f"Hauptmass — paarweises F1: {f1_mittel:.3f}")
if f1_mittel >= 0.9:
    print("  Einordnung: sehr gute Uebereinstimmung.")
elif f1_mittel >= 0.8:
    print("  Einordnung: gute Uebereinstimmung.")
else:
    print("  Einordnung: unzureichend — Richtlinie und Einweisung pruefen,")
    print("  bevor Tranche 2 begonnen wird.")
print(f"Tokenebene (nachrangig): Fleiss kappa = {fk_tok:.3f} "
      f"(beobachtet {pa_tok:.3f}, erwartet {pe_tok:.3f})")
print()
print("Fuer die Stoppregel gilt: Die Zielpraezision der Modellbewertung")
print("darf nicht enger angesetzt werden als das hier gemessene Agreement.")

print()
print("Alle bisherigen Laeufe dieser Tranche:")
for pfad in sorted(AUSGABE.glob(f"iaa_ner_t{TRANCHE}_lauf*.json")):
    d = json.loads(pfad.read_text(encoding="utf-8"))
    print(f"  Lauf {d['lauf']:>2} | {d['zeitpunkt']} | {d['etikett']:<22} | "
          f"F1 {d['hauptmass_paarweises_f1']['mittel']:.3f} | "
          f"strittig {d['strittige_spannen']}")
if LAUF_NR > 1:
    print()
    print("HINWEIS: Massgeblich ist der Lauf VOR der Adjudikation (Lauf 1).")

Geschrieben: ../05_ergebnisse/iaa/iaa_ner_t2_lauf01.json

Hauptmass — paarweises F1: 0.924
  Einordnung: sehr gute Uebereinstimmung.
Tokenebene (nachrangig): Fleiss kappa = 0.920 (beobachtet 0.992, erwartet 0.904)

Fuer die Stoppregel gilt: Die Zielpraezision der Modellbewertung
darf nicht enger angesetzt werden als das hier gemessene Agreement.

Alle bisherigen Laeufe dieser Tranche:
  Lauf  1 | 2026-09-13T11:33 | vor Adjudikation       | F1 0.924 | strittig 33
